In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=6, freq="D"),
    "A": [100, 102, 101, 105, 103, 108],
    "B": [200, 198, 202, 204, 201, 207]
})

df = df.set_index("Date")

df

,A,B
Date,,
2026-01-01,100,200
2026-01-02,102,198
2026-01-03,101,202
2026-01-04,105,204
2026-01-05,103,201
2026-01-06,108,207


# Session 1: Shape and axis

### DataFrame to NumPy | df.to_numpy()

In [4]:
prices = df.to_numpy(dtype=float)   # (6, 2)  -> (T, N)

prices

# default is int so use dtype=float

array([[100., 200.],
       [102., 198.],
       [101., 202.],
       [105., 204.],
       [103., 201.],
       [108., 207.]])

### Concept 1: Shape is a story

In [5]:
prices.shape  # (6, 2)

(6, 2)

6 rows, 2 columns

What happens to the labels?

**NumPy has no index or column names, only positions.**

In [6]:
dates = df.index.to_numpy()
dates

array(['2026-01-01T00:00:00.000000000', '2026-01-02T00:00:00.000000000',
       '2026-01-03T00:00:00.000000000', '2026-01-04T00:00:00.000000000',
       '2026-01-05T00:00:00.000000000', '2026-01-06T00:00:00.000000000'],
      dtype='datetime64[ns]')

In [7]:
dates.shape

(6,)

### Concept 2: Axis means “the dimension that disappears”

In [8]:
prices.mean()

152.58333333333334

In [9]:
prices.mean(axis=0)

# row wise operation, so mean for each column

array([103.16666667, 202.        ])

In [10]:
prices.mean(axis=1)

# column wise operation, so mean for each row

array([150. , 150. , 151.5, 154.5, 152. , 157.5])

The number you pass is the position in the shape that gets removed. For shape (T, N):

- **axis=0 removes T, so you get per-asset results (time series stats like mean, vol, max).**
- **axis=1 removes N, so you get per-date results (cross-sectional stats like the average across assets).**

### Concept 3: Indexing

In [11]:
df

,A,B
Date,,
2026-01-01,100,200
2026-01-02,102,198
2026-01-03,101,202
2026-01-04,105,204
2026-01-05,103,201
2026-01-06,108,207


In [12]:
prices

array([[100., 200.],
       [102., 198.],
       [101., 202.],
       [105., 204.],
       [103., 201.],
       [108., 207.]])

In [13]:
prices[2]

array([101., 202.])

In [14]:
prices[2].shape

(2,)

In [15]:
prices[2:3]

array([[101., 202.]])

In [16]:
prices[2:3].shape

(1, 2)

same: [rows, columns]  
: means everything

In [17]:
prices[:,0]

# all rows, 1st column

array([100., 102., 101., 105., 103., 108.])

In [18]:
prices[:,0].shape

(6,)

In [19]:
prices[:,0:1]

array([[100.],
       [102.],
       [101.],
       [105.],
       [103.],
       [108.]])

In [20]:
prices[:,0:1].shape

(6, 1)

(6, 1)   <- slice keeps the dimension

In [21]:
prices[1:4] # days 1 to 3, all assets    -> (3, 3)   <- end is exclusive

array([[102., 198.],
       [101., 202.],
       [105., 204.]])

In [22]:
prices[-1]

array([108., 207.])

In [23]:
prices[1:,:]

# drop first day

array([[102., 198.],
       [101., 202.],
       [105., 204.],
       [103., 201.],
       [108., 207.]])

In [24]:
prices[1:,:].shape # (5,2)

(5, 2)

In [25]:
prices[:-1,:]

# drop last day   

array([[100., 200.],
       [102., 198.],
       [101., 202.],
       [105., 204.],
       [103., 201.]])

In [26]:
prices[:-1,:].shape # (5,2)

(5, 2)

In [27]:
# ALL 

prices[2]          # day 2, all assets          -> (3,)
prices[:, 0]       # asset A, all days          -> (5,)
prices[:, 0:1]     # asset A, but keeps 2D      -> (5, 1)   <- slice keeps the dimension
prices[1:4]        # days 1 to 3, all assets    -> (3, 3)   <- end is exclusive
prices[-1]         # last day                   -> (3,)
prices[:, [0, 1]]  # assets A and B             -> (5, 2)
prices[1:, :]      # drop first day             -> (4, 3)
prices[:-1, :]     # drop last day              -> (4, 3)

array([[100., 200.],
       [102., 198.],
       [101., 202.],
       [105., 204.],
       [103., 201.]])

### Concept 4: Boolean masks (replace if inside loops)

In [28]:
prices

array([[100., 200.],
       [102., 198.],
       [101., 202.],
       [105., 204.],
       [103., 201.],
       [108., 207.]])

In [29]:
mask = prices > 102   
mask

array([[False,  True],
       [False,  True],
       [False,  True],
       [ True,  True],
       [ True,  True],
       [ True,  True]])

In [30]:
mask.sum()                  # total count of True

9

In [31]:
mask.sum(axis=0)            # per asset: how many days > 50   -> (3,)

array([3, 6])

In [32]:
prices[mask]  

array([200., 198., 202., 105., 204., 103., 201., 108., 207.])

In [33]:
# structure is lost

In [34]:
# To modify values while keeping the shape, assign through the mask on a copy:

p = prices.copy()
p[p < 102] = np.nan          # still (6, 3), with NaN where the condition was true

p

array([[ nan, 200.],
       [102., 198.],
       [ nan, 202.],
       [105., 204.],
       [103., 201.],
       [108., 207.]])

### Concept 5: NaNs

In [35]:
np.isnan(p)                 # (6, 3) boolean mask of NaN positions

array([[ True, False],
       [False, False],
       [ True, False],
       [False, False],
       [False, False],
       [False, False]])

In [36]:
np.isnan(p).sum(axis=0)     # NaN count per asset

array([2, 0])

### Shapes Test

In [37]:
prices

array([[100., 200.],
       [102., 198.],
       [101., 202.],
       [105., 204.],
       [103., 201.],
       [108., 207.]])

In [38]:
prices[:, 1] # all 6 rows and 1 column; but column has no dimension; shape -> (6,)

array([200., 198., 202., 204., 201., 207.])

In [39]:
prices[:, 1].shape

(6,)

In [40]:
prices[:, 0:1] # all 6 rows with 1 column, this has dimension shape -> (6,1)

array([[100.],
       [102.],
       [101.],
       [105.],
       [103.],
       [108.]])

In [41]:
prices[:, 0:1].shape

(6, 1)

In [42]:
prices[1, :] # 2nd row, all 2 columns without dimension shape -> (2,)

array([102., 198.])

In [43]:
prices[1, :].shape

(2,)

In [44]:
prices[0:1, :] # 1st row with dimension, all columns shape -> (1, 2)

array([[100., 200.]])

In [45]:
prices[0:1, :].shape

(1, 2)

Integer deletes its axis. Slice or list keeps its axis.

example:

In [46]:
prices[:,1]

array([200., 198., 202., 204., 201., 207.])

In [47]:
prices[:,1].shape  

(6,)

In [48]:
prices[:,0:1]

array([[100.],
       [102.],
       [101.],
       [105.],
       [103.],
       [108.]])

In [49]:
prices[:,0:1].shape

(6, 1)

example 2

In [50]:
prices[1,:]

array([102., 198.])

In [51]:
prices[1,:].shape

(2,)

In [52]:
prices[1:2,:]

array([[102., 198.]])

In [53]:
prices[1:2,:].shape

(1, 2)

example 3:

In [54]:
prices[2]

array([101., 202.])

In [55]:
prices[2].shape

(2,)

In [56]:
prices[:,1]

array([200., 198., 202., 204., 201., 207.])

In [57]:
prices[:,1].shape

(6,)

one is row another is column, but only 1d so elements is the shape

### Shape practise

In [58]:
import numpy as np

prices = np.array([
    [100., 50., 20.],   # day 0
    [102., 49., 21.],   # day 1
    [101., 51., 20.],   # day 2
    [105., 52., 22.],   # day 3
    [104., 50., 23.],   # day 4
])
# columns: asset A, asset B, asset C
print(prices.shape)   # (5, 3)  ->  (T, N)

(5, 3)


In [59]:
print(prices[0], prices[:3], prices[:, [0, 2]], prices.sum(axis=0), prices[1:, 1], prices[:, :1], sep="\n\n\n")


[100.  50.  20.]


[[100.  50.  20.]
 [102.  49.  21.]
 [101.  51.  20.]]


[[100.  20.]
 [102.  21.]
 [101.  20.]
 [105.  22.]
 [104.  23.]]


[512. 252. 106.]


[49. 51. 52. 50.]


[[100.]
 [102.]
 [101.]
 [105.]
 [104.]]


In [60]:
print(prices[0].shape, prices[:3].shape, prices[:, [0, 2]].shape, prices.sum(axis=0).shape, prices[1:, 1].shape, prices[:, :1].shape,  sep="\n")

(3,)
(3, 3)
(5, 2)
(3,)
(4,)
(5, 1)


### Axis practice

In [61]:
prices

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [62]:
# 1. The average price of each asset over time
prices.mean(axis=0)

# The average price across assets on each date
prices.mean(axis=1)

# The highest price each asset reached
prices.max(axis=0)

# The change from first day to last day for each asset
prices[-1] / prices[0] - 1

array([0.04, 0.  , 0.15])

### Dimensions | retaining dimensions **keepdims=True**

prices.mean(axis=1) gives (5,).   
If you later want to subtract that per-date average from prices (demeaning across assets), the shapes (5, 3) and (5,) won’t line up. The fix is:

In [63]:
prices.mean(axis=0).shape # 3 elements (3,)

(3,)

In [64]:
prices.mean(axis=0, keepdims=True)   # (1,3)  <- keeps the collapsed axis as length 1 | 1 row with 3 elements

array([[102.4,  50.4,  21.2]])

In [65]:
prices.mean(axis=0, keepdims=True).shape

(1, 3)

### Masks practise

In [66]:
prices

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [67]:
# How many entries in the whole matrix are greater than 50?
prices[prices>50] # (7,)

array([100., 102., 101.,  51., 105.,  52., 104.])

In [68]:
prices[prices>50] .shape

(7,)

In [69]:
# How many days was each asset above 50?
np.sum(prices>50, axis=0)

array([5, 2, 0])

In [70]:
(prices > 50).sum()          # () -> scalar: 7

7

In [71]:
# Booleans are True=1 and False=0, so summing a mask counts the Trues.
(prices > 50).sum(axis=0)    # (5,3) -> (3,): [5, 2, 0]

array([5, 2, 0])

This is a core backtesting idiom. For example, (signal > 0).sum(axis=0) gives the number of long days per asset.

In [72]:
# Make a copy of prices where every value below 21 is replaced with np.nan. Confirm the shape is still (5, 3).
prices_cpy = prices.copy()
prices_cpy

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [73]:
prices_cpy[prices_cpy<21] = np.nan

In [74]:
prices_cpy

array([[100.,  50.,  nan],
       [102.,  49.,  21.],
       [101.,  51.,  nan],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [75]:
prices_cpy.shape

(5, 3)

In [76]:
np.isnan(prices_cpy).sum() 

2

ask whether you were asked for the data or a statistic of the data.

### NaNs Practise

In [77]:
x = prices.copy(); x[2, 1] = np.nan

In [78]:
x

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  nan,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [79]:
x.mean()

nan

In [80]:
x.mean(axis=0)

array([102.4,   nan,  21.2])

In [81]:
# Compute the per-asset mean ignoring NaNs.



In [82]:
# Count the NaNs per asset.
np.isnan(x).sum(axis=0)

array([0, 1, 0])

**NumPy has a NaN-skipping twin for the common reductions: np.nansum, np.nanmean, np.nanstd, np.nanmax, np.nanmin**

In [83]:
np.nanmean(x, axis=0)

array([102.4 ,  50.25,  21.2 ])

### Backtest-realistic practise

In [84]:
prices

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [85]:
# 1. Compute the daily price change for all assets using two slices of prices, with no loops.

latest = prices[1:]
latest

array([[102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [86]:
rows = prices.shape[0]

prev = prices[:rows-1]
prev

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.]])

In [87]:
daily_price_change = latest / prev - 1
daily_price_change

array([[ 0.02      , -0.02      ,  0.05      ],
       [-0.00980392,  0.04081633, -0.04761905],
       [ 0.03960396,  0.01960784,  0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

In [88]:
daily_price_change.shape

(4, 3)

In [89]:
# Better:

today = prices[1:]            # (4,3)  days 1..4
prev  = prices[:-1]           # (4,3)  days 0..3
ret   = today / prev - 1      # (4,3)  row i = return from day i to day i+1

ret

array([[ 0.02      , -0.02      ,  0.05      ],
       [-0.00980392,  0.04081633, -0.04761905],
       [ 0.03960396,  0.01960784,  0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

**lesson: use `px[:-1]` to skip last row**

# Session 2: Returns and vectorized ops

In [90]:
prices

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

### Concept 1: Returns without loops

In [91]:
# ret = np.diff(prices) / prices[:-1]
# operands could not be broadcast together with shapes (5,2) (4,3) 

ret = np.diff(prices, axis=0) / prices[:-1]

ret

array([[ 0.02      , -0.02      ,  0.05      ],
       [-0.00980392,  0.04081633, -0.04761905],
       [ 0.03960396,  0.01960784,  0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

np.diff(..., axis=0) takes differences down the rows, so you get T-1 rows. Without axis=0 it would diff along columns, which is meaningless for this data.  
Always pass the axis.

#### axis = 0

In [92]:
np.diff(prices) # across columsn so always use axis=0 for rowwise

array([[-50., -30.],
       [-53., -28.],
       [-50., -31.],
       [-53., -30.],
       [-54., -27.]])

In [93]:
np.diff(prices, axis=0)

array([[ 2., -1.,  1.],
       [-1.,  2., -1.],
       [ 4.,  1.,  2.],
       [-1., -2.,  1.]])

### Concept 2: Cumulative returns and the equity curve

**1 + ret** gives growth factor  
apply cummulative product on growth factor to get cummulative returns like **(1+r1) x (1+r2) x (1+r3) ...** | `np.cumprod(1+r)`  
and in the end **subtract 1** to get the % change

In [94]:
1+ret # gives the growth factor: 1.02 multiplier is 2% increment in original

array([[1.02      , 0.98      , 1.05      ],
       [0.99019608, 1.04081633, 0.95238095],
       [1.03960396, 1.01960784, 1.1       ],
       [0.99047619, 0.96153846, 1.04545455]])

In [95]:
np.cumprod(1+ret, axis=0)

array([[1.02, 0.98, 1.05],
       [1.01, 1.02, 1.  ],
       [1.05, 1.04, 1.1 ],
       [1.04, 1.  , 1.15]])

In [96]:
eq = np.cumprod(1+ret, axis=0)

eq[-1]-1

array([0.04, 0.  , 0.15])

A small example shows why compounding matters:

In [97]:
r = np.array([0.10, -0.10])
np.cumprod(1 + r)    # [1.10, 0.99]  -> up 10% then down 10% is -1%, not 0%

array([1.1 , 0.99])

### Concept 3: Drawdown

**np.maximum.accumulate(eq)**

In [98]:
np.maximum.accumulate(prices) # for running max

array([[100.,  50.,  20.],
       [102.,  50.,  21.],
       [102.,  51.,  21.],
       [105.,  52.,  22.],
       [105.,  52.,  23.]])

In [99]:
peak = np.maximum.accumulate(eq)

peak

array([[1.02, 0.98, 1.05],
       [1.02, 1.02, 1.05],
       [1.05, 1.04, 1.1 ],
       [1.05, 1.04, 1.15]])

np.maximum.accumulate is the running maximum, and it’s the key to drawdown.  
On a matrix, pass axis=0 and it does every asset at once.

In [100]:
eq

array([[1.02, 0.98, 1.05],
       [1.01, 1.02, 1.  ],
       [1.05, 1.04, 1.1 ],
       [1.04, 1.  , 1.15]])

In [101]:
peak = np.maximum.accumulate(eq, axis=0)

peak

array([[1.02, 0.98, 1.05],
       [1.02, 1.02, 1.05],
       [1.05, 1.04, 1.1 ],
       [1.05, 1.04, 1.15]])

In [102]:
dd = eq/peak - 1

dd

array([[ 0.        ,  0.        ,  0.        ],
       [-0.00980392,  0.        , -0.04761905],
       [ 0.        ,  0.        ,  0.        ],
       [-0.00952381, -0.03846154,  0.        ]])

In [103]:
max_dd = dd.min(axis=0) 

max_dd

array([-0.00980392, -0.03846154, -0.04761905])

dd.min(axis=0) gives max drawdown per asset.

### Concept 4: np.where replaces if/else loops

In [104]:
signal = np.where(ret > 0, 1, -1)   # (4,3), +1 where return is positive, else -1

In [105]:
signal

array([[ 1, -1,  1],
       [-1,  1, -1],
       [ 1,  1,  1],
       [-1, -1,  1]])

The pattern is np.where(condition, value_if_true, value_if_false), and all three can be full arrays.  
One gotcha: NaN > 0 is False, so NaN returns silently become -1. If your data has NaNs, handle them explicitly.

### Concept 5: Lagging, the look-ahead guard

In [106]:
# we want today's signal

pos = signal[:-1]

pos

array([[ 1, -1,  1],
       [-1,  1, -1],
       [ 1,  1,  1]])

In [107]:
# with tomorrow's returns
ret[1:]

array([[-0.00980392,  0.04081633, -0.04761905],
       [ 0.03960396,  0.01960784,  0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

In [108]:
signal_ret = pos * ret[1:]

signal_ret

array([[-0.00980392, -0.04081633, -0.04761905],
       [-0.03960396,  0.01960784, -0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

The flow is: signal → lag → strategy returns → equity → drawdown.

#### Add a starting row of 1.0.  
Without it, the peak only starts from the first day’s equity, so a loss on day one never shows up as a drawdown.

In [109]:
equity = np.vstack([np.ones((1, 3)), eq])     # (4,3)

In [110]:
equity

array([[1.  , 1.  , 1.  ],
       [1.02, 0.98, 1.05],
       [1.01, 1.02, 1.  ],
       [1.05, 1.04, 1.1 ],
       [1.04, 1.  , 1.15]])

### Drills

In [111]:
prices

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

### Drill 1: Returns

#### **np.diff(px , axis=0)**

In [112]:
# Simple returns using np.diff.
np.diff(prices,axis=0) # (5,3) TO (4,3)

ret = np.diff(prices, axis = 0) / prices[:-1] # (4,3)

# Pad the simple returns with a NaN row at the top so the result is (5, 3).
np.vstack([np.zeros((1,3)),ret])

array([[ 0.        ,  0.        ,  0.        ],
       [ 0.02      , -0.02      ,  0.05      ],
       [-0.00980392,  0.04081633, -0.04761905],
       [ 0.03960396,  0.01960784,  0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

### Drill 2: Cummulative

In [113]:
# Equity curve from the simple returns, shape (4, 3).
1+ret
np.cumprod(1+ret, axis=0) # growth factor

array([[1.02, 0.98, 1.05],
       [1.01, 1.02, 1.  ],
       [1.05, 1.04, 1.1 ],
       [1.04, 1.  , 1.15]])

In [114]:
eq = np.cumprod(1+ret, axis=0)
eq

array([[1.02, 0.98, 1.05],
       [1.01, 1.02, 1.  ],
       [1.05, 1.04, 1.1 ],
       [1.04, 1.  , 1.15]])

In [115]:
# Total return per asset from it.
eq[-1]-1

array([0.04, 0.  , 0.15])

In [116]:
# Equity curve of shape (5, 3) with a first row of all 1.0 (stack ones on top of your Drill 2 curve).

# Pad the simple returns with a NaN row at the top so the result is (5, 3).
eq = np.cumprod(1+ret, axis=0) # growth factor
eq = np.vstack([np.ones((1,3)),eq])

np.vstack([np.full((1,3), np.nan), ret])   # what was required

eq

array([[1.  , 1.  , 1.  ],
       [1.02, 0.98, 1.05],
       [1.01, 1.02, 1.  ],
       [1.05, 1.04, 1.1 ],
       [1.04, 1.  , 1.15]])

### Drill 3: Drawdown

1) Equity curve of shape (5, 3) with a first row of all 1.0 (stack ones on top of your Drill 2 curve).  
2) Running peak and drawdown for every asset, no loops.  
3) Max drawdown per asset. Which asset had the worst one?

#### Running peak: `peak = np.maximum.accumulate(eq)`

In [117]:
# Running peak and drawdown for every asset, no loops.
peak = np.maximum.accumulate(eq)
peak # RUNNING PEAK

array([[1.  , 1.  , 1.  ],
       [1.02, 1.  , 1.05],
       [1.02, 1.02, 1.05],
       [1.05, 1.04, 1.1 ],
       [1.05, 1.04, 1.15]])

#### Running Drawdown: `dd = eq/peak - 1`

In [118]:
# Running Drawdown
dd = eq/peak - 1

dd

array([[ 0.        ,  0.        ,  0.        ],
       [ 0.        , -0.02      ,  0.        ],
       [-0.00980392,  0.        , -0.04761905],
       [ 0.        ,  0.        ,  0.        ],
       [-0.00952381, -0.03846154,  0.        ]])

In [119]:
# Max drawdown per asset. Which asset had the worst one?
max_drawdown = np.min(dd, axis=0) # (3,)
max_drawdown 

# 3rd one had the worst drawdown

array([-0.00980392, -0.03846154, -0.04761905])

### Drill 4: Signal and lag

In [120]:
# Signal: +1 if the day’s return was positive, else -1, shape (4, 3).

ret

array([[ 0.02      , -0.02      ,  0.05      ],
       [-0.00980392,  0.04081633, -0.04761905],
       [ 0.03960396,  0.01960784,  0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

#### np.where replaces if/else loops

In [121]:
# sig = 1 if ret>0 else -1

sig = np.where(ret>0,1,-1)

sig

array([[ 1, -1,  1],
       [-1,  1, -1],
       [ 1,  1,  1],
       [-1, -1,  1]])

In [122]:
# How many +1 days does each asset have?
np.where(sig==1,1,0).sum(axis=0)

array([2, 2, 3])

In [123]:
# Strategy returns for “trade tomorrow in the direction of today’s move”, shape (3, 3), and the mean strategy return per asset.
strat = sig[:-1]*ret[1:]
strat

array([[-0.00980392, -0.04081633, -0.04761905],
       [-0.03960396,  0.01960784, -0.1       ],
       [-0.00952381, -0.03846154,  0.04545455]])

In [124]:
strat.mean(axis=0)

array([-0.0196439 , -0.01989001, -0.03405483])

In [125]:
# why signal[:-1] pairs with ret[1:].
# -> to avoid look-ahead bias

# signal[i] is only known at the end of the day that return row i finishes, so it can only earn the next return, ret[i+1].

### Full mini-backtest chain

In [127]:
ret       = np.diff(prices, axis=0) / prices[:-1]               # (4,3)
sig       = np.where(ret > 0, 1, -1)                            # (4,3)
strat     = sig[:-1] * ret[1:]                                  # (3,3)
eq        = np.cumprod(1 + strat, axis=0)                       # (3,3)
eq        = np.vstack([np.ones((1, 3)), eq])                    # (4,3)
peak      = np.maximum.accumulate(eq, axis=0)                   # (4,3)
dd        = eq / peak - 1                                       # (4,3)
max_dd    = dd.min(axis=0)                                      # (3,)

# Session 3: Rolling Windows

In [132]:
# sum v cumsum
x = np.array([1, 2, 3, 4, 5])

x.sum()      # 15                     -> scalar
x.cumsum()   # [1, 3, 6, 10, 15]     -> (5,)  running total

array([ 1,  3,  6, 10, 15])

In [136]:
# Rolling windows

prices

array([[100.,  50.,  20.],
       [102.,  49.,  21.],
       [101.,  51.,  20.],
       [105.,  52.,  22.],
       [104.,  50.,  23.]])

In [134]:
from numpy.lib.stride_tricks import sliding_window_view as swv

In [137]:
windows = swv(prices, window_shape=3, axis=0)

windows

array([[[100., 102., 101.],
        [ 50.,  49.,  51.],
        [ 20.,  21.,  20.]],

       [[102., 101., 105.],
        [ 49.,  51.,  52.],
        [ 21.,  20.,  22.]],

       [[101., 105., 104.],
        [ 51.,  52.,  50.],
        [ 20.,  22.,  23.]]])

In [140]:
windows.shape

(3, 3, 3)

Why (3, 3, 3)?
- First 3: Number of rolling windows.
- Second 3: Number of assets.
- Third 3: Observations inside each window.

In [139]:
rolling_mean = windows.mean(axis=-1)

rolling_mean

array([[101.        ,  50.        ,  20.33333333],
       [102.66666667,  50.66666667,  21.        ],
       [103.33333333,  51.        ,  21.66666667]])

In [145]:
prices = np.array([
    [100,  50, 200,  80],  # Day 0
    [102,  52, 198,  82],  # Day 1
    [104,  51, 202,  84],  # Day 2
    [103,  53, 204,  83],  # Day 3
    [106,  55, 201,  85],  # Day 4
    [108,  54, 205,  87],  # Day 5
    [110,  54, 202,  83],  # Day 6
])

print(prices.shape)
# (6, 4)

(7, 4)


In [146]:
windows = swv(prices, window_shape=3, axis=0)
windows

array([[[100, 102, 104],
        [ 50,  52,  51],
        [200, 198, 202],
        [ 80,  82,  84]],

       [[102, 104, 103],
        [ 52,  51,  53],
        [198, 202, 204],
        [ 82,  84,  83]],

       [[104, 103, 106],
        [ 51,  53,  55],
        [202, 204, 201],
        [ 84,  83,  85]],

       [[103, 106, 108],
        [ 53,  55,  54],
        [204, 201, 205],
        [ 83,  85,  87]],

       [[106, 108, 110],
        [ 55,  54,  54],
        [201, 205, 202],
        [ 85,  87,  83]]])

In [147]:
windows.shape

(5, 4, 3)

5 windows  
4 columns
3 rows

4 assets  
3 observations per window

so we need the row axis value, therefore we use axis=-1

In [158]:

windows.shape
# (5, 4, 3)
#   |  |  |
#   |  |  └── Days inside each window (axis=2)
#   |  └───── Assets (axis=1)
#   └──────── Rolling windows (axis=0)

# (0, 1, 2) are the axis index in a way

(5, 4, 3)

In [150]:
rolling_mean = windows.mean(axis=2)
# We average the 3 days for each asset, independently.

In [152]:
# this is same as:
rolling_mean = windows.mean(axis=-1)

In [154]:
# Average across days within each window
windows.mean(axis=2)    # Shape: (4, 4)

array([[102.        ,  51.        , 200.        ,  82.        ],
       [103.        ,  52.        , 201.33333333,  83.        ],
       [104.33333333,  53.        , 202.33333333,  84.        ],
       [105.66666667,  54.        , 203.33333333,  85.        ],
       [108.        ,  54.33333333, 202.66666667,  85.        ]])

In [155]:

# Average across assets within each window
windows.mean(axis=1)    # Shape: (4, 3)


array([[107.5 , 108.5 , 110.25],
       [108.5 , 110.25, 110.75],
       [110.25, 110.75, 111.75],
       [110.75, 111.75, 113.5 ],
       [111.75, 113.5 , 112.25]])

In [156]:

# Average across all rolling windows
windows.mean(axis=0)    # Shape: (4, 3)

array([[103. , 104.6, 106.2],
       [ 52.2,  53. ,  53.4],
       [201. , 202. , 202.8],
       [ 82.8,  84.2,  84.4]])

**`axis = n` means eliminate dimension n by performing the operation along that dimension.**

- Axis 0 = windows
- Axis 1 = pairs
- Axis 2 = days inside each window  

We want to average the 3 days inside each window, so we eliminate axis 2.

# Session 4: Matrix

In [1]:
import numpy as np
import yfinance as yf          # pip install yfinance, if needed

tickers = ["URTH", "IXUS", "IYG", "GLD"]
w = np.array([0.4, 0.3, 0.2, 0.1])                 # (4,)  same order as tickers

df = yf.download(tickers, period="1mo", auto_adjust=True, progress=False)["Close"]
df = df[tickers].dropna().tail(11)                 # 11 prices -> 10 daily returns

prices = df.to_numpy(dtype=float)                  # (11, 4)
dates  = df.index.to_numpy()                       # (11,)
R      = prices[1:] / prices[:-1] - 1              # (10, 4)  daily returns

print(prices.shape, R.shape, w.sum())              # expect (11, 4) (10, 4) 1.0



(11, 4) (10, 4) 0.9999999999999999


In [2]:
prices

array([[205.1499939 ,  92.43000031,  93.30999756, 366.60998535],
       [206.22000122,  92.48999786,  93.58999634, 367.82000732],
       [207.58999634,  93.30000305,  93.5       , 372.27999878],
       [208.07000732,  93.83999634,  93.86000061, 373.51000977],
       [207.97000122,  93.75      ,  95.19999695, 368.57000732],
       [204.63999939,  92.34999847,  94.26000214, 355.77999878],
       [204.1499939 ,  92.55999756,  95.95999908, 361.6499939 ],
       [204.27999878,  92.12000275,  95.41999817, 359.26000977],
       [207.86999512,  93.94999695,  98.72000122, 372.48999023],
       [199.63000488,  90.44000244,  95.58000183, 375.48999023],
       [199.66999817,  90.55999756,  95.19999695, 372.51998901]])

In [3]:
df

Ticker,URTH,IXUS,IYG,GLD
Date,,,,
2026-10-23,205.149994,92.430000,93.309998,366.609985
2026-10-26,206.220001,92.489998,93.589996,367.820007
2026-10-27,207.589996,93.300003,93.500000,372.279999
2026-10-28,208.070007,93.839996,93.860001,373.510010
2026-10-29,207.970001,93.750000,95.199997,368.570007
2026-10-30,204.639999,92.349998,94.260002,355.779999
2026-11-02,204.149994,92.559998,95.959999,361.649994
2026-11-03,204.279999,92.120003,95.419998,359.260010
2026-11-04,207.869995,93.949997,98.720001,372.489990


In [4]:
w

array([0.4, 0.3, 0.2, 0.1])

In [5]:
R

array([[ 0.00521573,  0.00064911,  0.00300074,  0.00330057],
       [ 0.00664337,  0.00875776, -0.0009616 ,  0.01212547],
       [ 0.0023123 ,  0.00578771,  0.00385027,  0.00330399],
       [-0.00048064, -0.00095904,  0.01427654, -0.01322589],
       [-0.01601193, -0.01493335, -0.0098739 , -0.03470171],
       [-0.00239448,  0.00227395,  0.01803519,  0.01649895],
       [ 0.00063681, -0.00475362, -0.00562735, -0.00660856],
       [ 0.0175739 ,  0.01986533,  0.03458398,  0.03682564],
       [-0.03964011, -0.03736024, -0.03180712,  0.00805391],
       [ 0.00020034,  0.00132679, -0.00397578, -0.00790967]])

### Concept 1: @ multiplies matrices

`*` multiplies element by element. `@` does the “row times column, then add up” multiplication.

(a, b) @ (b, c)  ->  (a, c)      the two inner numbers must match

In [6]:
port = R @ w        # (10, 4) @ (4,) -> (10,)   one portfolio return per day

port

array([ 0.00321123,  0.0063049 ,  0.00376169,  0.00105275, -0.01632973,
        0.00498133, -0.00295769,  0.02358852, -0.03262015, -0.00110795])

In [8]:
# If the inner numbers don’t match, NumPy raises an error:
# R @ np.ones(3)      # (4,2) @ (3,) -> error, 2 vs 3

### Concept 2: Covariance and correlation matrices

In [16]:
cov  = np.cov(R, rowvar=False)        # (10, 4) -> (4, 4)

cov

array([[0.00023915, 0.0002318 , 0.00022736, 0.00011065],
       [0.0002318 , 0.00023523, 0.00023655, 0.00013129],
       [0.00022736, 0.00023655, 0.00031781, 0.00016782],
       [0.00011065, 0.00013129, 0.00016782, 0.00036847]])

In [17]:
corr = np.corrcoef(R, rowvar=False)   # (10, 4) -> (4, 4)

corr

array([[1.        , 0.97731155, 0.82469274, 0.37274729],
       [0.97731155, 1.        , 0.86516685, 0.44595285],
       [0.82469274, 0.86516685, 1.        , 0.49041817],
       [0.37274729, 0.44595285, 0.49041817, 1.        ]])

In [18]:
cov2  = np.cov(R)
cov2

array([[ 3.50652522e-06, -1.10939836e-06, -2.70683939e-06,
        -9.19663766e-07, -2.55737278e-06, -2.36946657e-06,
         3.75833157e-06, -3.96427009e-07,  2.19342047e-06,
        -1.54177381e-06],
       [-1.10939836e-06,  3.07865590e-05,  3.67141992e-07,
        -6.10356698e-05, -4.89608842e-05, -1.39423967e-05,
        -1.17286320e-06, -6.29813431e-06,  6.89456636e-05,
        -3.44751400e-06],
       [-2.70683939e-06,  3.67141992e-07,  2.13734752e-06,
         2.03062756e-06,  3.95947368e-06,  1.12777409e-07,
        -2.39310319e-06, -1.55409607e-06, -6.50514474e-06,
         2.06772105e-06],
       [-9.19663766e-07, -6.10356698e-05,  2.03062756e-06,
         1.26618931e-04,  1.10889542e-04,  1.38612981e-05,
         3.24243384e-06, -3.40812046e-06, -1.71844104e-04,
         1.51586299e-05],
       [-2.55737278e-06, -4.89608842e-05,  3.95947368e-06,
         1.10889542e-04,  1.18412733e-04, -3.21667066e-05,
         1.23133402e-05, -4.74496270e-05, -2.25015156e-04,
         3.

The gotcha: np.cov assumes rows are variables by default. Your data has dates in rows and assets in columns, so you must pass rowvar=False. Without it you get a (4, 4) matrix of nonsense. I’d write rowvar=False every time.

### Concept 3: Portfolio variance in one line

In [19]:
var = w @ cov @ w 

var

0.0002196702748781655

In [21]:
vol = np.sqrt(var)       # daily portfolio volatility

vol

0.01482127777481299

In [22]:
annualized_vol = vol*np.sqrt(250)

annualized_vol

0.23434497801220613

### Practise

In [23]:
import numpy as np
import pandas as pd
import yfinance as yf

tickers = ["URTH", "IXUS", "IYG", "GLD"]
w = np.array([0.4, 0.3, 0.2, 0.1])

all_tickers = tickers + ["SPY"]

prices = yf.download(
    all_tickers,
    period="1mo",
    auto_adjust=True,
    progress=False
)["Close"]

prices = prices[all_tickers].dropna().tail(11)

print(prices.tail())
print(prices.shape)

Ticker            URTH       IXUS        IYG         GLD         SPY
Date                                                                
2026-11-02  204.149994  92.559998  95.959999  361.649994  752.099976
2026-11-03  204.279999  92.120003  95.419998  359.260010  754.349976
2026-11-04  207.869995  93.949997  98.720001  372.489990  766.809998
2026-11-05  199.630005  90.440002  95.580002  375.489990  733.000000
2026-11-06  199.669998  90.559998  95.199997  372.519989  735.289978
(11, 5)


In [29]:
print(type(prices), type(w))

<class 'pandas.core.frame.DataFrame'> <class 'numpy.ndarray'>


#### Calculate daily returns — Pandas

In [30]:
returns = prices.pct_change().dropna()

asset_returns = returns[tickers]
market_returns = returns["SPY"]

In [31]:
asset_returns.shape

(10, 4)

In [32]:
market_returns.shape

(10,)

In [34]:
asset_returns

Ticker,URTH,IXUS,IYG,GLD
Date,,,,
2026-10-26,0.005216,0.000649,0.003001,0.003301
2026-10-27,0.006643,0.008758,-0.000962,0.012125
2026-10-28,0.002312,0.005788,0.003850,0.003304
2026-10-29,-0.000481,-0.000959,0.014277,-0.013226
2026-10-30,-0.016012,-0.014933,-0.009874,-0.034702
2026-11-02,-0.002394,0.002274,0.018035,0.016499
2026-11-03,0.000637,-0.004754,-0.005627,-0.006609
2026-11-04,0.017574,0.019865,0.034584,0.036826
2026-11-05,-0.039640,-0.037360,-0.031807,0.008054


In [33]:
market_returns

Date
2026-10-26    0.007106
2026-10-27    0.005605
2026-10-28    0.003443
2026-10-29   -0.003820
2026-10-30   -0.017729
2026-11-02   -0.004790
2026-11-03    0.002992
2026-11-04    0.016518
2026-11-05   -0.044092
2026-11-06    0.003124
Name: SPY, dtype: float64

In [35]:
returns[["SPY"]]

Ticker,SPY
Date,
2026-10-26,0.007106
2026-10-27,0.005605
2026-10-28,0.003443
2026-10-29,-0.003820
2026-10-30,-0.017729
2026-11-02,-0.004790
2026-11-03,0.002992
2026-11-04,0.016518
2026-11-05,-0.044092


Why Pandas?

Because pct_change() operates naturally on time-series columns. It also preserves dates and column names.

#### Calclate Beta for each asset

In [37]:
cov_matrix = returns.cov()

cov_matrix

Ticker,URTH,IXUS,IYG,GLD,SPY
Ticker,,,,,
URTH,0.000239,0.000232,0.000227,0.000111,0.000260
IXUS,0.000232,0.000235,0.000237,0.000131,0.000248
IYG,0.000227,0.000237,0.000318,0.000168,0.000231
GLD,0.000111,0.000131,0.000168,0.000368,0.000106
SPY,0.000260,0.000248,0.000231,0.000106,0.000287


returns.cov() calculates the covariance matrix automatically.

In [39]:
market_variance = returns["SPY"].var()

market_variance

0.0002867896624287031

In [40]:
asset_cov = cov_matrix.loc[tickers, "SPY"]

asset_cov

Ticker
URTH    0.000260
IXUS    0.000248
IYG     0.000231
GLD     0.000106
Name: SPY, dtype: float64

In [42]:
betas = asset_cov / market_variance

betas

Ticker
URTH    0.905795
IXUS    0.866011
IYG     0.804219
GLD     0.369639
Name: SPY, dtype: float64

#### Calculate portfolio beta

In [43]:
asset_returns

Ticker,URTH,IXUS,IYG,GLD
Date,,,,
2026-10-26,0.005216,0.000649,0.003001,0.003301
2026-10-27,0.006643,0.008758,-0.000962,0.012125
2026-10-28,0.002312,0.005788,0.003850,0.003304
2026-10-29,-0.000481,-0.000959,0.014277,-0.013226
2026-10-30,-0.016012,-0.014933,-0.009874,-0.034702
2026-11-02,-0.002394,0.002274,0.018035,0.016499
2026-11-03,0.000637,-0.004754,-0.005627,-0.006609
2026-11-04,0.017574,0.019865,0.034584,0.036826
2026-11-05,-0.039640,-0.037360,-0.031807,0.008054


In [44]:
w

array([0.4, 0.3, 0.2, 0.1])

In [46]:
pf_returns = asset_returns.to_numpy() @ w
pf_returns

array([ 0.00321123,  0.0063049 ,  0.00376169,  0.00105275, -0.01632973,
        0.00498133, -0.00295769,  0.02358852, -0.03262015, -0.00110795])

In [48]:
portfolio_returns = pd.Series(
    pf_returns,
    index=asset_returns.index
)

portfolio_returns

Date
2026-10-26    0.003211
2026-10-27    0.006305
2026-10-28    0.003762
2026-10-29    0.001053
2026-10-30   -0.016330
2026-11-02    0.004981
2026-11-03   -0.002958
2026-11-04    0.023589
2026-11-05   -0.032620
2026-11-06   -0.001108
dtype: float64

Pandas for beta

In [49]:
portfolio_beta = portfolio_returns.cov(market_returns) / market_returns.var()

print("Portfolio Beta:", portfolio_beta)

Portfolio Beta: 0.8199289289115471


Numpy for beta

In [56]:
portfolio_beta = (
    np.cov(portfolio_returns, market_returns)[0, 1] 
    / np.var(market_returns, ddof=1)
)

print("Portfolio Beta:", portfolio_beta)

Portfolio Beta: 0.8199289289115471


If you use ddof=0 for both covariance and variance, beta remains identical because both are divided by N.

Operation	& Purpose  
`.to_numpy()`	Convert Pandas to NumPy  
`@`	Matrix multiplication  
`np.cov()`	Covariance matrix  
`np.sqrt()`	Square root  
`np.dot()`	Dot product  
`np.corrcoef()`	Correlation matrix  

In [50]:
import numpy as np

# 1. Convert Pandas DataFrame to NumPy array
asset_returns_np = asset_returns.to_numpy()

# 2. Matrix multiplication (@) — Calculate daily portfolio returns
portfolio_returns = asset_returns_np @ w

# Shapes:
# asset_returns_np = (10, 4)
# w                = (4,)
# portfolio_returns = (10,)


# 3. Portfolio variance — w'Σw
# Extract covariance matrix of the 4 portfolio assets
asset_cov_matrix = np.cov(asset_returns_np, rowvar=False)

portfolio_variance = w.T @ asset_cov_matrix @ w

# Portfolio volatility
portfolio_volatility = np.sqrt(portfolio_variance)


# 5. Correlation matrix
correlation_matrix = np.corrcoef(returns.to_numpy(), rowvar=False)


# print("Portfolio Returns:", portfolio_returns)
print("Portfolio Volatility:", portfolio_volatility)
print("Correlation Matrix:\n", correlation_matrix)

Portfolio Volatility: 0.01482127777481299
Correlation Matrix:
 [[1.         0.97731155 0.82469274 0.37274729 0.99192079]
 [0.97731155 1.         0.86516685 0.44595285 0.9562231 ]
 [0.82469274 0.86516685 1.         0.49041817 0.76395867]
 [0.37274729 0.44595285 0.49041817 1.         0.32610616]
 [0.99192079 0.9562231  0.76395867 0.32610616 1.        ]]


#### For 50 day Beta calculation with some stocks not having enough data points

In [60]:
WINDOW = 5
MIN_OBS = 3

In [61]:
# Align stock and market returns by date
data = returns.join(market_returns.rename("MARKET"), how="inner").tail(WINDOW)

In [62]:
data

,URTH,IXUS,IYG,GLD,SPY,MARKET
Date,,,,,,
2026-11-02,-0.002394,0.002274,0.018035,0.016499,-0.004790,-0.004790
2026-11-03,0.000637,-0.004754,-0.005627,-0.006609,0.002992,0.002992
2026-11-04,0.017574,0.019865,0.034584,0.036826,0.016518,0.016518
2026-11-05,-0.039640,-0.037360,-0.031807,0.008054,-0.044092,-0.044092
2026-11-06,0.000200,0.001327,-0.003976,-0.007910,0.003124,0.003124


In [65]:
w

array([0.4, 0.3, 0.2, 0.1])

In [66]:
tickers = ["URTH", "IXUS", "IYG", "GLD"]

w = np.array([0.4, 0.3, 0.2, 0.1])

weights = pd.Series(w, index=tickers)

In [67]:
weights

URTH    0.4
IXUS    0.3
IYG     0.2
GLD     0.1
dtype: float64

In [70]:
betas = {}

for ticker in weights.index:
    stock = data[ticker].dropna()

    market = data.loc[stock.index, "MARKET"] 
    # It's simply selecting the market returns for the same dates that the stock has available.

    if len(stock) >= MIN_OBS:
        betas[ticker] = stock.cov(market) / market.var()
    else:
        betas[ticker] = np.nan

betas = pd.Series(betas)
# Why? So Pandas can align beta values with portfolio weights by ticker name:

portfolio_beta = (weights * betas).sum()

portfolio_beta = np.dot(weights.to_numpy(), betas.to_numpy())
# numpy version; FASTER

print("Individual Betas:\n", betas)
print("\nPortfolio Beta:", portfolio_beta)

Individual Betas:
 URTH    0.909134
IXUS    0.875409
IYG     0.903856
GLD     0.177129
dtype: float64

Portfolio Beta: 0.8247603770828156


#### Matrix Multiplication is NOT commutative | A @ B != B @ A

For pf variance:  
**Order matters** because matrix **multiplication is not commutative**:  
`w.T @ cov @ w`

A @ B != B @ A